# 00 — Download e Exploração do Dataset SelfRegulationSCP1

Classificação **binária multivariada**: 6 canais EEG (slow cortical potentials),
comprimento 896, 2 classes (`negativity`/`positivity`). Dataset do UEA archive (via `aeon`).

EEG é o domínio canônico de wavelets — sinais não-estacionários e multi-escala.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
sys.path.insert(0, 'config')
from experiment_config import (
    SCP1_CONFIG, DATA_DIR, RESULTS_DIR, SEED,
    WAVELET_CONFIG, LEARNED_WAVELET_CONFIG, DL_TRAINING_CONFIG,
    ML_MODELS_CONFIG, ML_SEARCH_CONFIG, build_param_dist, N_JOBS_QUARTER,
)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(SEED)
print('Dataset:', SCP1_CONFIG['dataset_name'], '| classes:', SCP1_CONFIG['n_classes'],
      '| seq_len:', SCP1_CONFIG['sequence_length'], '| canais:', SCP1_CONFIG['n_features'])

## 1. Download e Preparação

In [ ]:
from src.data_loader import SCP1DataLoader
loader = SCP1DataLoader(data_dir=DATA_DIR, random_seed=SEED)
data = loader.prepare(val_fraction=SCP1_CONFIG['val_fraction'])
X_train, y_train = data['X_train'], data['y_train']
X_val,   y_val   = data['X_val'],   data['y_val']
X_test,  y_test  = data['X_test'],  data['y_test']
print('shapes:', X_train.shape, X_val.shape, X_test.shape)

## 2. Distribuição de Classes

In [ ]:
from src.visualization import ExperimentVisualizer
viz = ExperimentVisualizer()
for split, y in [('train', y_train), ('val', y_val), ('test', y_test)]:
    u, c = np.unique(y, return_counts=True)
    print(f'{split:5s}:', dict(zip(u.tolist(), c.tolist())))
viz.plot_class_distribution(y_train, title='Distribuição (treino)'); plt.show()

## 3. Visualização dos Sinais (6 canais EEG por amostra)

In [ ]:
viz.plot_samples(X_train, y_train, n_per_class=3, title='Amostras SelfRegulationSCP1 (EEG)'); plt.show()

## 4. Análise Espectral (Welch, por canal)

In [ ]:
from scipy.signal import welch
fig, ax = plt.subplots(figsize=(10, 4))
for c in range(X_train.shape[2]):
    f, pxx = welch(X_train[:, :, c].mean(axis=0))
    ax.semilogy(f, pxx, label=f'canal {c}')
ax.set_xlabel('Frequência normalizada'); ax.set_ylabel('PSD'); ax.legend(); ax.grid(alpha=.3)
plt.title('Densidade espectral média por canal'); plt.show()

## 5. Decomposição Wavelet (exemplo, db4)

In [ ]:
import pywt
sig = X_train[0, :, 0]
coeffs = pywt.wavedec(sig, WAVELET_CONFIG['wavelet_type'], level=WAVELET_CONFIG['decomposition_level'])
viz.plot_wavelet_decomposition(sig, coeffs[0], coeffs[1:], title='Decomposição Wavelet (canal 0)'); plt.show()

## 6. Extração de Features Wavelet (multivariada — para ML)

In [ ]:
from src.feature_extraction import WaveletFeatureExtractor
extractor = WaveletFeatureExtractor(wavelet=WAVELET_CONFIG['wavelet_type'], level=WAVELET_CONFIG['decomposition_level'])
Xtr_feat = extractor.extract_features(X_train)
print('features por amostra:', Xtr_feat.shape[1])
np.save(DATA_DIR / 'X_train_wavelet_features.npy', Xtr_feat)
np.save(DATA_DIR / 'X_val_wavelet_features.npy',   extractor.extract_features(X_val))
np.save(DATA_DIR / 'X_test_wavelet_features.npy',  extractor.extract_features(X_test))
print('✓ features wavelet salvas')

## 7. Resumo

Dados prontos em `data/`. Próximos: **01** (ML), **02–04** (DL via fila), **05** (comparação), **06** (filtros).